In [0]:
# Definir las rutas
from pyspark.sql import functions as F

# Ruta de entrada
source_path = "s3://moises-sales-data/landing/sales/"

# Ubicaciones persistentes para Auto Loader
schema_path = "s3://moises-sales-data/shema/"
checkpoint_path = "s3://moises-sales-data/checkpoints/"
checkpoint_write_path = "s3://moises-sales-data/checkpoints/bronze_write/"

# Tabla Delta de destino
target_table = "aws_s3.ventas.bronze_ventas_raw"

In [0]:
# Leer los archivos con Auto Loader
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("sep", ",")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

# Agregar trazabilidad
df_bronze = (
    df_bronze
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# Preview verificado en ejecución anterior

In [0]:
# Guardar los datos en Delta Lake
query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_write_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()